In [ ]:
!pip install openai pandas

In [ ]:
API_KEY = "" # Copy your OpenAI API key

In [ ]:
import time
import pandas as pd
from openai import OpenAI

# =====================================================
# CONFIGURATION
# =====================================================
INPUT_FILE = "/content/Train_64_maxSample_1pertemp.csv"
OUTPUT_FILE = "Train_64_maxSample_1pertemp_output.csv"
STATS_FILE = "Train_64_maxSample_1pertemp_avg_time.csv"

MODEL_NAME = "gpt-5.2"
# Initialize OpenAI client
client = OpenAI(api_key=API_KEY)

# =====================================================
# FUNCTION TO CALL GPT-5.2
# =====================================================
def extract_template(log_event):

# System and User Prompt for Zero-shot Inference
    system_prompt = (
        "You are a log parsing assistant."
        "Your task is to identify variable elements within the logs, generalize these elements, and construct a template that represents the structure of these log messages. "
        "Please, follow the instruction and return an accurate response."
    )

    user_prompt = (
        f"Extract the template of the following log message. Replace all variable elements "
        f"with '<*>'. Do not provide any explanation, only return the template.\n\n{log_event}"
    )

# System and User Prompt for Few-shot Inference

#     system_prompt = (
#         "You are a log parsing assistant. "
#         "Your task is to identify variable elements within the logs, "
#         "generalize these elements, and construct a template that represents "
#         "the structure of these log messages. "
#         "Please, follow the instruction and return an accurate response."
#     )

#     user_prompt = f"""
# ### Instruction:
# Extract the template for the following log message. Replace any variable element with the placeholder '<*>'.
# Do not provide any explanation, only return the template.

# ### Example 1:
# Log message: Error = , LOGIN chdir(/home/spelce1/UMT2K/umt2k/ckpt_umt2k_src/TEST/NEW_TEST) failed: No such file at /10.10.34.20:56374 point [/TEST/NEW_TEST], connect to proxy proxy.cse.cuhk.edu.hk:5070 to renew session (0x14f05578bd8001b)
# Extracted template: Error = , LOGIN chdir(<*>) failed: No such file at <*>:<*> point [<*>], connect to proxy <*>:<*> to renew session (<*>)

# ### Example 2:
# Log message: ciod: In packet from nodes 91.0 and node-234 (R62-M1-Nf-C:J03-U11), message code 2 is not 3 or 4294967295 (softheader=003b005b 00030000 00000001 00000000)
# Extracted template: ciod: In packet from nodes <*> and <*> (<*>:<*>), message code <*> is not <*> or <*> (softheader=<*> <*> <*> <*>)

# ### Input:
# {log_event}

# ### Response:
# """

    start_time = time.time()

    try:
        response = client.responses.create(
            model=MODEL_NAME,
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.0,
            max_output_tokens=1000,
        )

        end_time = time.time()
        inference_time = end_time - start_time

        # GPT-5.2 output handling
        template = response.output_text.strip()

        input_tokens = response.usage.input_tokens
        output_tokens = response.usage.output_tokens

        return template, inference_time, input_tokens, output_tokens

    except Exception as e:
        return f"Error: {e}", 0, 0, 0


In [ ]:
def process_csv(input_path, output_path, stats_path):
    print(f"Loading input file: {input_path}")

    try:
        df = pd.read_csv(input_path)
    except Exception as e:
        print(f"Failed to read {input_path}: {e}")
        return

    if df.empty:
        print("The input CSV file is empty. Nothing to process.")
        return

    required_cols = ["Content", "Dataset"]
    for col in required_cols:
        if col not in df.columns:
            print(f"Missing required column: '{col}'")
            return

    has_event_template = "EventTemplate" in df.columns

    print(f"Loaded {len(df)} log entries.\n")

    gpt_templates, gpt_times, in_tokens, out_tokens = [], [], [], []

    for i, row in df.iterrows():
        log_event = str(row["Content"])
        dataset_name = str(row["Dataset"])
        event_template = str(row["EventTemplate"]) if has_event_template else "N/A"

        print(f"[{i+1}/{len(df)}] Dataset: {dataset_name}")
        print(f"   Log Event: {log_event}")

        template, t, in_tok, out_tok = extract_template(log_event)

        print(f"   Ground Truth Template: {event_template}")
        print(f"   GPT Extracted Template: {template}")
        print(f"   Inference Time: {t:.3f} sec\n", flush=True)

        gpt_templates.append(template)
        gpt_times.append(t)
        in_tokens.append(in_tok)
        out_tokens.append(out_tok)

    # Add results to dataframe
    df["GPT_Temp_1"] = gpt_templates
    df["GPT_Time_1"] = gpt_times
    df["GPT_In_Tokens_1"] = in_tokens
    df["GPT_Out_Tokens_1"] = out_tokens

    # Save detailed results
    try:
        df.to_csv(output_path, index=False)
        print(f"\nSaved full results to: {output_path}")
    except Exception as e:
        print(f"Failed to save {output_path}: {e}")

    # Compute per-dataset statistics
    stats = (
        df.groupby("Dataset")
        .agg(
            Total_Logs=("Content", "count"),
            Total_Time=("GPT_Time_1", "sum"),
            Avg_Time_per_Log=("GPT_Time_1", "mean"),
        )
        .reset_index()
    )

    # Save stats
    try:
        stats.to_csv(stats_path, index=False)
        print(f"Saved dataset-level stats to: {stats_path}")
    except Exception as e:
        print(f"Failed to save {stats_path}: {e}")

    print("\n===Dataset-Level Summary ===")
    print(stats.to_string(index=False), flush=True)
    print("\nProcessing complete.\n")


# =====================================================
# Main Function
# =====================================================
if __name__ == "__main__":
    process_csv(INPUT_FILE, OUTPUT_FILE, STATS_FILE)